In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# ✈️ Travel Planner with Multi-Stage Checkpoints

## What We're Building

An interactive travel planner that asks for user approval at each stage:

1. **Gather preferences** → ⏸ user confirms
2. **Search destinations** → ⏸ user picks
3. **Build itinerary** → ⏸ user approves
4. **Generate final plan** with booking checklist

## Architecture

```
┌────────────┐    ┌────────────┐    ┌────────────┐    ┌────────────┐
│  Gather    │──▶ │  Search    │──▶ │  Build     │──▶ │  Final     │
│  Prefs     │    │  Dests     │    │  Itinerary │    │  Plan      │
└────────────┘    └────────────┘    └────────────┘    └────────────┘
      ⏸                ⏸                 ⏸
  user confirms   user picks dest   user approves
```

## Key LangGraph Concept: Multiple Interrupt Points

Unlike prior notebooks with a single `interrupt_before`, this workflow has
**three interrupt gates**. Each pause lets the human review, modify state,
and resume. This demonstrates iterative human-AI collaboration.

## Stack
- **LangGraph** — multi-interrupt checkpointed workflow
- **Ollama** — local LLM

## Step 1 — Imports & Setup

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_ollama import ChatOllama
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOllama(model="qwen3.5:9b", temperature=0.3)
print("All imports successful!")

## Step 2 — State Definition

In [ ]:
class TravelState(TypedDict):
    # Input
    raw_request: str              # User's initial travel request
    # Stage 1: Preferences
    parsed_preferences: str       # Structured preferences extracted from request
    prefs_confirmed: bool         # User confirmed preferences
    # Stage 2: Destinations
    destination_options: str      # Suggested destinations with pros/cons
    chosen_destination: str       # User's pick
    # Stage 3: Itinerary
    draft_itinerary: str          # Day-by-day plan
    itinerary_feedback: str       # User's modifications
    itinerary_approved: bool      # User approved
    # Stage 4: Final
    final_plan: str               # Complete plan with booking checklist


print("TravelState defined with 4-stage fields")

## Step 3 — Define Nodes

### Node 1: Parse Preferences

In [ ]:
prefs_prompt = ChatPromptTemplate.from_template(
    """Parse this travel request into structured preferences.

Request: {request}

Extract:
- TRAVELERS: Number, ages, relationships (solo, couple, family)
- DATES: Departure and return dates (or duration)
- BUDGET: Total budget or per-person budget
- STYLE: Adventure, relaxation, cultural, mixed
- CLIMATE: Warm, cold, mild, no preference
- MUST-HAVES: Activities/experiences they specifically want
- DEAL-BREAKERS: Things to avoid
- DIETARY: Any food restrictions
- MOBILITY: Any accessibility needs
- DEPARTURE CITY: Where they're flying from

Structured preferences:"""
)


def gather_preferences(state: TravelState) -> dict:
    print("📋 Node: gather_preferences")
    chain = prefs_prompt | llm | StrOutputParser()
    prefs = chain.invoke({"request": state["raw_request"]})
    return {"parsed_preferences": prefs}

### Node 2: Search Destinations

In [ ]:
dest_prompt = ChatPromptTemplate.from_template(
    """Based on these travel preferences, suggest 3 destination options.

Preferences:
{prefs}

For each destination provide:
- DESTINATION: City/region, Country
- WHY IT FITS: How it matches their preferences
- ESTIMATED COST: Rough budget breakdown (flights, hotel, activities)
- BEST TIME: Whether their dates align with good weather/events
- CONS: Potential downsides or mismatches
- SAMPLE DAY: One exciting day of activities

Rank them: Option 1 = best match, Option 3 = stretch pick.

Destination options:"""
)


def search_destinations(state: TravelState) -> dict:
    print("🔍 Node: search_destinations")
    chain = dest_prompt | llm | StrOutputParser()
    options = chain.invoke({"prefs": state["parsed_preferences"]})
    return {"destination_options": options}

### Node 3: Build Itinerary

In [ ]:
itinerary_prompt = ChatPromptTemplate.from_template(
    """Create a day-by-day travel itinerary for this trip.

Destination: {destination}
Preferences: {prefs}

Create a detailed itinerary:
- Day-by-day plan (morning / afternoon / evening)
- Include specific venues, restaurants, and activities
- Add travel time estimates between activities
- Note reservation requirements (book in advance vs walk-in)
- Include 1 rest/flex period (travelers get tired!)
- Budget estimate per day

Day-by-day itinerary:"""
)


def build_itinerary(state: TravelState) -> dict:
    print("🗓️ Node: build_itinerary")
    chain = itinerary_prompt | llm | StrOutputParser()
    itinerary = chain.invoke({
        "destination": state["chosen_destination"],
        "prefs": state["parsed_preferences"]
    })
    return {"draft_itinerary": itinerary}

### Node 4: Generate Final Plan

In [ ]:
final_prompt = ChatPromptTemplate.from_template(
    """Create the final travel plan document.

Destination: {destination}
Itinerary: {itinerary}
Traveler Feedback: {feedback}

Create a comprehensive travel plan including:

1. TRIP OVERVIEW (destination, dates, travelers, total budget)
2. PRE-TRIP CHECKLIST
   - Documents needed (passport, visa, insurance)
   - Bookings to make (flights, hotels, activities)
   - Packing essentials
3. DAY-BY-DAY ITINERARY (incorporating any feedback)
4. BUDGET BREAKDOWN
   - Flights, accommodation, food, activities, transport, contingency
5. EMERGENCY INFO
   - Embassy contact, local emergency numbers, nearest hospital
6. USEFUL PHRASES (if foreign-language destination)

Final travel plan:"""
)


def generate_final_plan(state: TravelState) -> dict:
    print("📄 Node: generate_final_plan")
    chain = final_prompt | llm | StrOutputParser()
    plan = chain.invoke({
        "destination": state["chosen_destination"],
        "itinerary": state["draft_itinerary"],
        "feedback": state.get("itinerary_feedback", "No modifications requested")
    })
    return {"final_plan": plan}


print("All 4 nodes defined")

## Step 4 — Build Graph with 3 Interrupt Points

The key: `interrupt_before` accepts a list of node names.
The graph pauses BEFORE executing those nodes, giving the user
a chance to inspect and modify state.

In [ ]:
workflow = StateGraph(TravelState)

workflow.add_node("gather_preferences", gather_preferences)
workflow.add_node("search_destinations", search_destinations)
workflow.add_node("build_itinerary", build_itinerary)
workflow.add_node("generate_final_plan", generate_final_plan)

workflow.set_entry_point("gather_preferences")
workflow.add_edge("gather_preferences", "search_destinations")
workflow.add_edge("search_destinations", "build_itinerary")
workflow.add_edge("build_itinerary", "generate_final_plan")
workflow.add_edge("generate_final_plan", END)

memory = MemorySaver()

# Three interrupt points — pause before search, itinerary, and final plan
app = workflow.compile(
    checkpointer=memory,
    interrupt_before=["search_destinations", "build_itinerary", "generate_final_plan"]
)

config = {"configurable": {"thread_id": "trip-planning-2024"}}
print("Travel planner compiled with 3 interrupt points")

## Step 5 — Stage 1: Gather & Confirm Preferences

The first `invoke()` runs `gather_preferences` then pauses
BEFORE `search_destinations`.

In [ ]:
travel_request = """
My wife and I want to celebrate our 10th anniversary.
We're thinking 7 days sometime in October.
Budget is around $5,000 total.
We love food and wine — would love a culinary-focused trip.
We enjoy walking tours and historical sites but nothing too strenuous.
We're flying from New York.
My wife is vegetarian. Neither of us speaks any language besides English.
We'd prefer somewhere warm-ish but not tropical.
"""

result = app.invoke(
    {
        "raw_request": travel_request,
        "parsed_preferences": "", "prefs_confirmed": False,
        "destination_options": "", "chosen_destination": "",
        "draft_itinerary": "", "itinerary_feedback": "", "itinerary_approved": False,
        "final_plan": "",
    },
    config
)

print("\n⏸️ PAUSED before search_destinations")
print("\n📋 Parsed Preferences:")
print(result["parsed_preferences"])

### User Reviews & Confirms Preferences

The user can modify the parsed preferences before continuing.
Here we simulate the user confirming with a small correction.

In [ ]:
# Simulate user confirming preferences (with a tweak)
app.update_state(
    config,
    {
        "prefs_confirmed": True,
        # User adds a note: they'd also like to visit a vineyard
        "parsed_preferences": result["parsed_preferences"] + "\n\nADDITIONAL NOTE: Would love to visit a vineyard or winery."
    }
)
print("✅ User confirmed preferences (added vineyard note)")

## Step 6 — Stage 2: Search Destinations & Pick

Resume with `invoke(None, config)` — runs `search_destinations`
then pauses BEFORE `build_itinerary`.

In [ ]:
result = app.invoke(None, config)

print("\n⏸️ PAUSED before build_itinerary")
print("\n🌍 Destination Options:")
print(result["destination_options"])

### User Picks a Destination

In [ ]:
# Simulate user choosing a destination
app.update_state(
    config,
    {"chosen_destination": "Tuscany, Italy — We love the idea of rolling hills, wine, and Italian food!"}
)
print("✅ User chose: Tuscany, Italy")

## Step 7 — Stage 3: Build & Approve Itinerary

Resume again — runs `build_itinerary` then pauses BEFORE `generate_final_plan`.

In [ ]:
result = app.invoke(None, config)

print("\n⏸️ PAUSED before generate_final_plan")
print("\n🗓️ Draft Itinerary:")
print(result["draft_itinerary"])

### User Reviews Itinerary & Provides Feedback

In [ ]:
# Simulate user feedback on the itinerary
app.update_state(
    config,
    {
        "itinerary_feedback": (
            "Love the overall plan! A few tweaks:\n"
            "- Day 3 looks too packed — can we swap one activity for free time?\n"
            "- Add a cooking class if possible (we've always wanted to make pasta from scratch)\n"
            "- Make sure all restaurants have vegetarian options clearly noted"
        ),
        "itinerary_approved": True,
    }
)
print("✅ User approved itinerary with feedback")

## Step 8 — Stage 4: Generate Final Travel Plan

Last resume — runs `generate_final_plan` and reaches END.

In [ ]:
result = app.invoke(None, config)

print("\n" + "="*60)
print("📄 FINAL TRAVEL PLAN")
print("="*60)
print(result["final_plan"])

## Step 9 — Inspect Full Workflow History

The checkpoint history shows every pause and resume point.

In [ ]:
print("🕰️ Checkpoint History (most recent first):")
print("-" * 50)
for i, snapshot in enumerate(app.get_state_history(config)):
    meta = snapshot.metadata or {}
    source = meta.get("source", "?")
    step = meta.get("step", "?")
    writes = meta.get("writes", {})
    nodes = list(writes.keys()) if writes else []
    print(f"  Step {step:>3} | source={source:8s} | wrote={nodes}")
    if i >= 20:
        print("  ...")
        break

print(f"\n✅ Workflow complete — 4 invoke() calls, 3 human review gates")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Multiple interrupts** | `interrupt_before=["node_a", "node_b", "node_c"]` |
| **State inspection** | Read `result["field"]` at each pause to show user |
| **State modification** | `app.update_state(config, {"field": "value"})` |
| **Resume** | `app.invoke(None, config)` continues from last pause |
| **Checkpoint history** | `app.get_state_history(config)` shows full audit trail |

### The Multi-Interrupt Pattern

```python
# 1. First invoke → runs nodes until first interrupt
result = app.invoke(initial_state, config)

# 2. User reviews, updates state
app.update_state(config, {"user_choice": "Option A"})

# 3. Resume → runs until next interrupt
result = app.invoke(None, config)

# 4. Repeat until END
```

## 🔧 Extensions

- **Real API integration**: Plug in flight/hotel APIs (Amadeus, Booking.com)
- **Budget optimizer**: Re-route to cheaper alternatives if over budget
- **Weather check**: Conditional node that verifies weather for chosen dates
- **Group planning**: Multiple travelers review and vote on destinations